# Skeleton-EAA-Pose Kaggle Pipeline

Full Kaggle version of the Colab pipeline. Google Drive remains the durable storage; Kaggle working storage is used only for lightweight metadata/tracks/logs and the current staged video batch. RGB videos are copied in waves capped by `KAGGLE_MAX_BATCH_GB` and deleted after each wave.


## 1. Install Repo and Dependencies


In [ ]:
REPO_URL = 'https://github.com/tuan8p/Skeleton-EAA-Pose.git'
REPO_PATH = '/kaggle/working/Skeleton-EAA-Pose'
INSTALL_MMPOSE_STACK = True  # Required for Step 2B. Set False if only running Step 2A.

from pathlib import Path
import os
import subprocess
import sys

repo_path = Path(REPO_PATH)
if repo_path.exists():
    subprocess.run(['git', '-C', REPO_PATH, 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_PATH], check=True)

os.chdir(REPO_PATH)
print('cwd:', os.getcwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

if INSTALL_MMPOSE_STACK:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'openmim'], check=True)
    subprocess.run(['mim', 'install', '-q', 'mmengine', 'mmcv', 'mmdet', 'mmpose'], check=True)


## 2. Configure rclone non-interactively

Kaggle cells cannot reliably accept interactive terminal input, so do not run `rclone config` here. Create `rclone.conf` outside Kaggle once, then provide it to this notebook using one of these non-interactive methods:

1. Recommended: create a private Kaggle Dataset containing `rclone.conf` and attach it to this notebook.
2. Alternative: create a Kaggle Secret named `RCLONE_CONF` containing the full text of `rclone.conf`.

The config must contain a remote section named `[gdrive]`, or you must change `RCLONE_REMOTE_NAME` below to match your section name.


In [ ]:
# Remote name used everywhere below. It must match the section name in rclone.conf, e.g. [gdrive].
RCLONE_REMOTE_NAME = 'gdrive'

from pathlib import Path
import os
import shutil
import subprocess
import textwrap

def run(cmd, check=True, capture_output=False):
    print('$', ' '.join(cmd))
    return subprocess.run(cmd, check=check, text=True, capture_output=capture_output)

# Install rclone once per Kaggle session if missing.
if shutil.which('rclone') is None:
    run(['bash', '-lc', 'curl https://rclone.org/install.sh | sudo bash'])
run(['rclone', 'version'])

config_dir = Path.home() / '.config' / 'rclone'
config_path = config_dir / 'rclone.conf'
config_dir.mkdir(parents=True, exist_ok=True)

# Option 1: Kaggle Secret named RCLONE_CONF containing the full rclone.conf text.
if not config_path.exists():
    try:
        from kaggle_secrets import UserSecretsClient
        secret_conf = UserSecretsClient().get_secret('RCLONE_CONF')
        if secret_conf and '[' in secret_conf and 'type = drive' in secret_conf:
            config_path.write_text(secret_conf, encoding='utf-8')
            print('Wrote rclone.conf from Kaggle Secret RCLONE_CONF')
    except Exception as exc:
        print('No usable Kaggle Secret RCLONE_CONF:', exc)

# Option 2: attached private Dataset containing a file named rclone.conf.
if not config_path.exists():
    candidates = sorted(Path('/kaggle/input').glob('**/rclone.conf'))
    if candidates:
        shutil.copy2(candidates[0], config_path)
        print('Copied rclone.conf from', candidates[0])

if not config_path.exists():
    raise FileNotFoundError(textwrap.dedent(f'''
    Missing {config_path}. Kaggle cannot complete interactive rclone config inside a cell.

    Do this once outside Kaggle:
    1. In Colab or local terminal, run: rclone config
    2. Create a Google Drive remote named: {RCLONE_REMOTE_NAME}
    3. Copy the generated rclone.conf into Kaggle by either:
       - attaching a private Kaggle Dataset containing rclone.conf, or
       - creating a Kaggle Secret named RCLONE_CONF with the full file contents.

    Then rerun this cell.
    '''))

print('Using rclone config:', config_path)
remotes = run(['rclone', 'listremotes'], check=False, capture_output=True)
print(remotes.stdout)
if f'{RCLONE_REMOTE_NAME}:\n' not in remotes.stdout and f'{RCLONE_REMOTE_NAME}:\r\n' not in remotes.stdout:
    raise RuntimeError(f'rclone.conf exists, but it has no remote named {RCLONE_REMOTE_NAME}. Check the section name, e.g. [gdrive].')

result = subprocess.run(['rclone', 'lsd', f'{RCLONE_REMOTE_NAME}:'], text=True)
if result.returncode != 0:
    raise RuntimeError(f'Remote {RCLONE_REMOTE_NAME}: exists but is not usable. The token may be expired or the wrong account was authorized.')
print(f'Remote {RCLONE_REMOTE_NAME}: is ready.')


## 3. Dataset and Remote Paths


In [ ]:
DATASET = 'pku_v1'  # 'pku_v1' | 'pku_v2' | 'tsu'

if DATASET == 'pku_v1':
    CONFIG_FILE = f'{REPO_PATH}/configs/pku_v1.yaml'
elif DATASET == 'pku_v2':
    CONFIG_FILE = f'{REPO_PATH}/configs/pku_v2.yaml'
elif DATASET == 'tsu':
    CONFIG_FILE = f'{REPO_PATH}/configs/tsu.yaml'
else:
    raise ValueError(DATASET)

# Edit these paths to the exact locations under your rclone remote.
KAGGLE_REMOTE_NAME = RCLONE_REMOTE_NAME
KAGGLE_REMOTE_VIDEO_DIR = 'ĐACN-TN_datasets/ĐATN/rawdatasets/videos/PKUv1'
KAGGLE_REMOTE_SEGMENTS_DIR = 'ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/PKU/Label_PKUMMDv1_daily'
KAGGLE_REMOTE_ACTIONS_PATH = 'ĐACN-TN_datasets/ĐATN/rawdatasets/skeletons/PKU/Actions_daily.csv'
KAGGLE_REMOTE_OUT_DIR = 'ĐACN-TN_datasets/ĐATN/rawdatasets/PKU_MMD_v1/samples_npy1'

KAGGLE_REMOTE_OUT = f'{KAGGLE_REMOTE_NAME}:{KAGGLE_REMOTE_OUT_DIR}'.rstrip('/')
KAGGLE_ACTIONS_ARG = ''
if KAGGLE_REMOTE_ACTIONS_PATH:
    KAGGLE_ACTIONS_ARG = f'--remote-actions-path "{KAGGLE_REMOTE_ACTIONS_PATH}"'

print('Dataset:', DATASET)
print('Config:', CONFIG_FILE)
print('Remote video dir:', KAGGLE_REMOTE_VIDEO_DIR)
print('Remote segments dir:', KAGGLE_REMOTE_SEGMENTS_DIR)
print('Remote output dir:', KAGGLE_REMOTE_OUT_DIR)


## 3.1. Validate remote paths

Run this before Step 2A. It fails early if a Drive path is wrong, which is much nicer than discovering it after launching a batch.


In [ ]:
import subprocess

def check_remote_dir(label, path):
    remote = f'{KAGGLE_REMOTE_NAME}:{path}'
    print(f'Checking {label}: {remote}')
    result = subprocess.run(['rclone', 'lsf', remote], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Missing or inaccessible remote path: {remote}')
    print('  OK, first entries:')
    print('\n'.join(result.stdout.splitlines()[:5]) or '  <empty>')

check_remote_dir('videos', KAGGLE_REMOTE_VIDEO_DIR)
check_remote_dir('segments', KAGGLE_REMOTE_SEGMENTS_DIR)
if KAGGLE_REMOTE_ACTIONS_PATH:
    remote_action = f'{KAGGLE_REMOTE_NAME}:{KAGGLE_REMOTE_ACTIONS_PATH}'
    print('Checking actions file:', remote_action)
    result = subprocess.run(['rclone', 'lsf', remote_action], text=True, capture_output=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Missing or inaccessible actions file: {remote_action}')
print('Remote paths look usable.')


## 4. Runtime Options


In [ ]:
TRACKING_MODEL = 'yolo26m.pt'
TRACKING_TRACKER = 'bytetrack.yaml'
TRACK_START_INDEX = 0
TRACK_END_INDEX = None
RUN_ALL_TRACKS = True
TRACK_LIMIT = 3
TRACK_QC_RETRY_LIMIT = None
TRACK_QC_MAX_INTERP_GAP = 10
TRACK_QC_MAX_INTERP_ANCHOR_DISTANCE = None  # None means max_gap + 2

RUN_ALL_POSE = True
POSE_LIMIT = 3
POSE_START_INDEX = 0
POSE_END_INDEX = None
POSE_NUM_PERSONS = 1

# Kaggle working/output storage is limited. Keep staged video waves <= 15 GB.
KAGGLE_WORK_DIR = '/kaggle/working/eaa_pose_kaggle'
KAGGLE_STAGE_DIR = '/kaggle/working/eaa_pose_kaggle/staged_videos'
KAGGLE_WORKERS = 2
KAGGLE_DEVICES = '0,1'
KAGGLE_BATCH_SIZE_TRACKS = 8
KAGGLE_BATCH_SIZE_POSE = 4
KAGGLE_MAX_BATCH_GB = 15

print(f'Tracking: {TRACKING_MODEL}, {TRACKING_TRACKER}')
print(f'Track range: [{TRACK_START_INDEX}:{TRACK_END_INDEX}] run_all={RUN_ALL_TRACKS} limit={TRACK_LIMIT}')
print(f'Track QC max interpolation gap: {TRACK_QC_MAX_INTERP_GAP}')
print(f'Track QC max anchor distance: {TRACK_QC_MAX_INTERP_ANCHOR_DISTANCE}')
print(f'Pose range: [{POSE_START_INDEX}:{POSE_END_INDEX}] run_all={RUN_ALL_POSE} limit={POSE_LIMIT}')
print('Stage dir:', KAGGLE_STAGE_DIR, 'max batch GB:', KAGGLE_MAX_BATCH_GB)


## 5. Optional Module 1 Note

For Kaggle, keep Module 1 outputs (`Label_PKUMMDv1_daily`, `Actions_daily.csv`) already prepared on Drive and point `KAGGLE_REMOTE_SEGMENTS_DIR` / `KAGGLE_REMOTE_ACTIONS_PATH` to them. Running Module 1 on Kaggle is possible but usually unnecessary because it does not need GPU and may require syncing many skeleton/label files.


## 6. Step 2A: YOLO26 + ByteTrack Person Tracking

Existing `tracks/<video_id>_tracks.json` files are synced from Drive first, so completed videos are skipped.


In [ ]:
track_scope_arg = '--all' if RUN_ALL_TRACKS else f'--limit {TRACK_LIMIT}'
track_range_arg = f'--start-index {TRACK_START_INDEX}'
if TRACK_END_INDEX is not None:
    track_range_arg += f' --end-index {TRACK_END_INDEX}'

cmd = f'''python -m eaa_pose.run_kaggle_tracks_from_drive \
    --config "{CONFIG_FILE}" \
    --remote-name "{KAGGLE_REMOTE_NAME}" \
    --remote-video-dir "{KAGGLE_REMOTE_VIDEO_DIR}" \
    --remote-segments-dir "{KAGGLE_REMOTE_SEGMENTS_DIR}" \
    {KAGGLE_ACTIONS_ARG} \
    --remote-out-dir "{KAGGLE_REMOTE_OUT_DIR}" \
    --work-dir "{KAGGLE_WORK_DIR}" \
    --stage-dir "{KAGGLE_STAGE_DIR}" \
    --workers {KAGGLE_WORKERS} \
    --devices "{KAGGLE_DEVICES}" \
    --batch-size {KAGGLE_BATCH_SIZE_TRACKS} \
    --max-batch-gb {KAGGLE_MAX_BATCH_GB} \
    --tracking-model "{TRACKING_MODEL}" \
    --tracking-tracker "{TRACKING_TRACKER}" \
    {track_range_arg} \
    {track_scope_arg}'''
print(cmd)
!{cmd}


## 7. Inspect Track Stats


In [ ]:
import json
from pathlib import Path

track_stats_path = Path(KAGGLE_WORK_DIR) / 'outputs' / 'track_stats.json'
if track_stats_path.exists():
    track_stats = json.loads(track_stats_path.read_text(encoding='utf-8'))
    print(json.dumps(track_stats.get('status_counts_in_action', {}), indent=2, ensure_ascii=False))
    for status, items in track_stats.get('videos_by_status', {}).items():
        if items:
            print(status, len(items), items[:3])
else:
    print('Missing:', track_stats_path)


## 8. Step 2A_QC_1: Retry No-Detection Track Videos

This syncs current `tracks/*_tracks.json` from Drive, stages only videos that still contain in-action `no_detection`, then overwrites their track JSON files on Drive.


In [ ]:
retry_limit_arg = '' if TRACK_QC_RETRY_LIMIT is None else f'--limit {TRACK_QC_RETRY_LIMIT}'

cmd = f'''python -m eaa_pose.run_kaggle_track_qc_retry_from_drive \
    --config "{CONFIG_FILE}" \
    --remote-name "{KAGGLE_REMOTE_NAME}" \
    --remote-video-dir "{KAGGLE_REMOTE_VIDEO_DIR}" \
    --remote-segments-dir "{KAGGLE_REMOTE_SEGMENTS_DIR}" \
    {KAGGLE_ACTIONS_ARG} \
    --remote-out-dir "{KAGGLE_REMOTE_OUT_DIR}" \
    --work-dir "{KAGGLE_WORK_DIR}" \
    --stage-dir "{KAGGLE_STAGE_DIR}" \
    --workers {KAGGLE_WORKERS} \
    --devices "{KAGGLE_DEVICES}" \
    --batch-size {KAGGLE_BATCH_SIZE_TRACKS} \
    --max-batch-gb {KAGGLE_MAX_BATCH_GB} \
    --tracking-model "{TRACKING_MODEL}" \
    --tracking-tracker "{TRACKING_TRACKER}" \
    {retry_limit_arg}'''
print(cmd)
!{cmd}


## 9. Step 2A_QC_2: Interpolate Short No-Detection Gaps

This step does not need RGB videos. It syncs tracks from Drive, runs interpolation locally, then syncs updated tracks and `track_stats_qc.json` back to Drive.


In [ ]:
from pathlib import Path

LOCAL_OUT_DIR = Path(KAGGLE_WORK_DIR) / 'outputs'
LOCAL_TRACKS_DIR = LOCAL_OUT_DIR / 'tracks'
LOCAL_TRACKS_DIR.mkdir(parents=True, exist_ok=True)
track_stats_qc_path = LOCAL_OUT_DIR / 'track_stats_qc.json'

!rclone copy "{KAGGLE_REMOTE_OUT}/tracks" "{LOCAL_TRACKS_DIR}" --include "*_tracks.json"

anchor_arg = '' if TRACK_QC_MAX_INTERP_ANCHOR_DISTANCE is None else f'--max-anchor-distance {TRACK_QC_MAX_INTERP_ANCHOR_DISTANCE}'

cmd = f'''python -m eaa_pose.run_track_qc_interpolate \
    --config "{CONFIG_FILE}" \
    --out-dir "{LOCAL_OUT_DIR}" \
    --max-gap "{TRACK_QC_MAX_INTERP_GAP}" \
    {anchor_arg}'''
print(cmd)
!{cmd}

!rclone copy "{LOCAL_TRACKS_DIR}" "{KAGGLE_REMOTE_OUT}/tracks" --include "*_tracks.json"
!rclone copyto "{track_stats_qc_path}" "{KAGGLE_REMOTE_OUT}/track_stats_qc.json"


## 10. Inspect Track QC Stats


In [ ]:
track_qc_stats_path = Path(KAGGLE_WORK_DIR) / 'outputs' / 'track_stats_qc.json'
if track_qc_stats_path.exists():
    track_qc_stats = json.loads(track_qc_stats_path.read_text(encoding='utf-8'))
    print(json.dumps(track_qc_stats.get('status_counts_in_action', {}), indent=2, ensure_ascii=False))
    for status, items in track_qc_stats.get('videos_by_status', {}).items():
        if items:
            print(status, len(items), items[:3])
else:
    print('Missing:', track_qc_stats_path)


## 11. Step 2B: RTMW3D Pose From Track JSON

This stages pending RGB videos in batches, copies their matching track JSON files into each worker output, runs pose on two GPUs, syncs generated `.npy` samples and QC reports back to Drive, then deletes local worker outputs. Existing sample `.npy` files on Drive are used for skip planning.


In [ ]:
pose_scope_arg = '' if RUN_ALL_POSE else f'--limit {POSE_LIMIT}'
pose_range_arg = f'--start-index {POSE_START_INDEX}'
if POSE_END_INDEX is not None:
    pose_range_arg += f' --end-index {POSE_END_INDEX}'

cmd = f'''python -m eaa_pose.run_kaggle_pose_from_drive \
    --config "{CONFIG_FILE}" \
    --remote-name "{KAGGLE_REMOTE_NAME}" \
    --remote-video-dir "{KAGGLE_REMOTE_VIDEO_DIR}" \
    --remote-segments-dir "{KAGGLE_REMOTE_SEGMENTS_DIR}" \
    {KAGGLE_ACTIONS_ARG} \
    --remote-out-dir "{KAGGLE_REMOTE_OUT_DIR}" \
    --work-dir "{KAGGLE_WORK_DIR}" \
    --stage-dir "{KAGGLE_STAGE_DIR}" \
    --workers {KAGGLE_WORKERS} \
    --devices "{KAGGLE_DEVICES}" \
    --batch-size {KAGGLE_BATCH_SIZE_POSE} \
    --max-batch-gb {KAGGLE_MAX_BATCH_GB} \
    --num-persons {POSE_NUM_PERSONS} \
    {pose_range_arg} \
    {pose_scope_arg}'''
print(cmd)
!{cmd}


## 12. Inspect Metadata, Pose Stats, and QC Reports


In [ ]:
LOCAL_OUT_DIR = Path(KAGGLE_WORK_DIR) / 'outputs'
metadata_path = LOCAL_OUT_DIR / 'metadata.json'
pose_stats_path = LOCAL_OUT_DIR / 'pose_stats.json'

for path in [metadata_path, pose_stats_path]:
    if path.exists():
        payload = json.loads(path.read_text(encoding='utf-8'))
        print('---', path.name)
        print(json.dumps(payload, indent=2, ensure_ascii=False)[:4000])
    else:
        print('Missing:', path)

qc_dir = LOCAL_OUT_DIR / 'qc'
if qc_dir.exists():
    print('Local QC files:', len(list(qc_dir.glob('*.json'))))
else:
    print('Local QC dir may be empty because worker outputs are cleaned after sync.')


## 13. Verify One SkateFormer Sample


In [ ]:
import glob
import numpy as np

local_samples = sorted(glob.glob(str(Path(KAGGLE_WORK_DIR) / 'outputs' / '*.npy')))
if local_samples:
    sample = local_samples[0]
    arr = np.load(sample)
    print(sample, arr.shape, arr.dtype)
else:
    print('No local .npy samples retained; they were synced to Drive and worker outputs were cleaned.')
    !rclone lsf "{KAGGLE_REMOTE_OUT}" --files-only | head


## 14. Disk Check


In [ ]:
!du -sh /kaggle/working/eaa_pose_kaggle || true
!find /kaggle/working/eaa_pose_kaggle -maxdepth 2 -type d -print | sort
